# Unidad 2 — Tarea 4: Reporte de fusión y agrupación

**Asignatura:** Programación avanzada para ciencia de datos  
**Entorno:** Python, Pandas y Jupyter Notebook  
**Objetivo:** Fusionar ventas, productos y sucursales; validar las relaciones y analizar ingresos por sucursal, categoría y mes.

> Los archivos originales no se modifican. Las transformaciones se aplican a copias en memoria.

## 1. Relaciones entre las tres tablas

```mermaid
erDiagram
    VENTAS }o--|| PRODUCTOS : producto_id
    VENTAS }o--|| SUCURSALES : sucursal_id
    VENTAS {
        int venta_id PK
        string producto_id FK
        string sucursal_id FK
        date fecha
        number cantidad
        number precio_unitario
    }
    PRODUCTOS {
        string producto_id PK
        string nombre
        string categoria
        number precio_lista
    }
    SUCURSALES {
        string sucursal_id PK
        string nombre
        string municipio
        object contacto
    }
```

Cada venta corresponde a un producto y una sucursal; cada producto o sucursal puede aparecer en muchas ventas. Las claves primarias del catálogo son `producto_id` y `sucursal_id`; `venta_id` identifica cada transacción.

## 2. Carga y diagnóstico inicial

In [1]:
from pathlib import Path
import pandas as pd

# Localizar los datos tanto si el notebook está en src como en la raíz del proyecto.
actual = Path.cwd()
candidatas = [actual / "datos", actual / "src" / "datos", actual.parent / "datos"]
carpeta_datos = next((p for p in candidatas if (p / "ventas_sucio.csv").exists()), None)
if carpeta_datos is None:
    raise FileNotFoundError("No se encontró datos/ventas_sucio.csv. Coloca los tres archivos en src/datos/.")

# Leer los tres archivos sin modificar los originales.
ventas = pd.read_csv(carpeta_datos / "ventas_sucio.csv")
productos = pd.read_csv(carpeta_datos / "productos.csv")
sucursales = pd.read_json(carpeta_datos / "sucursales.json")

# Verificar dimensiones y estructura.
for nombre, tabla in [("Ventas", ventas), ("Productos", productos), ("Sucursales", sucursales)]:
    print(f"{nombre}: {tabla.shape[0]} filas, {tabla.shape[1]} columnas")
    print("Columnas:", tabla.columns.tolist())


Ventas: 612 filas, 6 columnas
Columnas: ['venta_id', 'fecha', 'sucursal_id', 'producto_id', 'cantidad', 'precio_unitario']
Productos: 12 filas, 4 columnas
Columnas: ['producto_id', 'nombre', 'categoria', 'precio_lista']
Sucursales: 4 filas, 4 columnas
Columnas: ['sucursal_id', 'nombre', 'municipio', 'contacto']


In [2]:
# Diagnóstico de nulos y duplicados antes de la limpieza.
print("Nulos en ventas por columna:")
print(ventas.isna().sum().to_string())
print("\nFilas exactamente duplicadas:", int(ventas.duplicated().sum()))
print("Productos duplicados por clave:", int(productos["producto_id"].duplicated().sum()))
print("Sucursales duplicadas por clave:", int(sucursales["sucursal_id"].duplicated().sum()))
print("Variantes de sucursal en ventas:", ventas["sucursal_id"].nunique())


Nulos en ventas por columna:
venta_id            0
fecha               0
sucursal_id         0
producto_id         0
cantidad           26
precio_unitario    18

Filas exactamente duplicadas: 12
Productos duplicados por clave: 0
Sucursales duplicadas por clave: 0
Variantes de sucursal en ventas: 13


## 3. Preparación de datos y criterios

- Se eliminan únicamente **filas completamente idénticas**, para evitar contar dos veces la misma venta.
- Se estandarizan identificadores de sucursal con `strip()` y `upper()`.
- Se convierten precios con `$` y cantidades con sufijo `pz` a valores numéricos.
- Se interpretan fechas en formatos ISO y `día/mes/año`.
- Los importes de ventas sin cantidad o precio se dejan como **NA**; no se inventan valores.
- Las cantidades extremas se conservan y se señalan como limitación: la tarea se centra en las fusiones y no hay evidencia suficiente para sustituirlas.

In [3]:
# Copias de trabajo: preservan los archivos de origen.
v = ventas.drop_duplicates().copy()
p = productos.copy()
s = sucursales.copy()

# Unificar identificadores para garantizar las correspondencias.
for tabla, columna in [(v, "sucursal_id"), (s, "sucursal_id"), (v, "producto_id"), (p, "producto_id")]:
    tabla[columna] = tabla[columna].astype("string").str.strip().str.upper()

# Convertir cantidades como "9 pz" y "7.0" a números.
v["cantidad"] = pd.to_numeric(
    v["cantidad"].astype("string").str.replace(r"\s*pz\s*$", "", regex=True, case=False).str.strip(),
    errors="coerce"
)

# Retirar símbolo de moneda y separadores de miles del precio.
v["precio_unitario"] = pd.to_numeric(
    v["precio_unitario"].astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip(),
    errors="coerce"
)

# Interpretar fechas ISO y fechas escritas como DD/MM/AAAA.
fechas_texto = v["fecha"].astype("string").str.strip()
fechas_iso = pd.to_datetime(fechas_texto.where(fechas_texto.str.contains(r"^\d{4}-\d{2}-\d{2}$")), errors="coerce", format="%Y-%m-%d")
fechas_dmy = pd.to_datetime(fechas_texto.where(~fechas_texto.str.contains(r"^\d{4}-\d{2}-\d{2}$")), errors="coerce", format="%d/%m/%Y")
v["fecha"] = fechas_iso.fillna(fechas_dmy)

print("Filas originales:", len(ventas), "| Filas sin duplicados exactos:", len(v))
print("Fechas no interpretadas:", int(v["fecha"].isna().sum()))
print("Cantidades no disponibles:", int(v["cantidad"].isna().sum()))
print("Precios no disponibles:", int(v["precio_unitario"].isna().sum()))
print("Claves de sucursal sin catálogo:", set(v["sucursal_id"].dropna()) - set(s["sucursal_id"].dropna()))


Filas originales: 612 | Filas sin duplicados exactos: 600
Fechas no interpretadas: 0
Cantidades no disponibles: 25
Precios no disponibles: 18
Claves de sucursal sin catálogo: set()


## 4. Fusiones con `validate` y comprobación de filas

In [4]:
# Evitar columnas ambiguas renombrando los campos de nombre.
p = p.rename(columns={"nombre": "nombre_producto"})
s = s.rename(columns={"nombre": "nombre_sucursal"})

# Comprobar que las claves de los catálogos sean únicas.
assert p["producto_id"].is_unique, "Hay productos repetidos en el catálogo"
assert s["sucursal_id"].is_unique, "Hay sucursales repetidas en el catálogo"

# Primera fusión: cada venta puede referirse a un único producto.
ventas_productos = v.merge(p, on="producto_id", how="left", validate="many_to_one", indicator=True)
print("Antes de fusionar:", len(v), "| Después de productos:", len(ventas_productos))
print("Correspondencias de productos:", ventas_productos["_merge"].value_counts().to_dict())
assert len(ventas_productos) == len(v), "La fusión de productos cambió las filas"
assert (ventas_productos["_merge"] == "both").all(), "Hay ventas sin producto asociado"
ventas_productos = ventas_productos.drop(columns="_merge")

# Segunda fusión: cada venta puede referirse a una única sucursal.
ventas_integradas = ventas_productos.merge(s, on="sucursal_id", how="left", validate="many_to_one", indicator=True)
print("Antes de fusionar:", len(ventas_productos), "| Después de sucursales:", len(ventas_integradas))
print("Correspondencias de sucursales:", ventas_integradas["_merge"].value_counts().to_dict())
assert len(ventas_integradas) == len(v), "La fusión de sucursales cambió las filas"
assert (ventas_integradas["_merge"] == "both").all(), "Hay ventas sin sucursal asociada"
ventas_integradas = ventas_integradas.drop(columns="_merge")

# Calcular el importe; si faltan cantidad o precio, el importe queda nulo.
ventas_integradas["importe"] = ventas_integradas["cantidad"] * ventas_integradas["precio_unitario"]
ventas_integradas["mes"] = ventas_integradas["fecha"].dt.to_period("M").astype("string")
print("\nTabla integrada:", ventas_integradas.shape)
display(ventas_integradas.head())


Antes de fusionar: 600 | Después de productos: 600
Correspondencias de productos: {'both': 600, 'left_only': 0, 'right_only': 0}
Antes de fusionar: 600 | Después de sucursales: 600
Correspondencias de sucursales: {'both': 600, 'left_only': 0, 'right_only': 0}

Tabla integrada: (600, 14)


,venta_id,fecha,sucursal_id,producto_id,cantidad,precio_unitario,nombre_producto,categoria,precio_lista,nombre_sucursal,municipio,contacto,importe,mes
0,1001,2026-01-14,S03,P003,7.0,30.37,Leche 1 L,Lácteos,29,Bosque de Tlahuac,Bosque,"{'gerente': 'A. Ruiz', 'tel': '55-1111-0003'}",212.59,2026-01
1,1002,2026-04-08,S02,P009,10.0,34.41,Azúcar 1 kg,Abarrotes,33,La Turba,Turba,"{'gerente': 'R. Díaz', 'tel': '55-1111-0002'}",344.1,2026-04
2,1003,2026-06-15,S02,P012,1.0,40.98,Papel higiénico 4 pz,Limpieza,42,La Turba,Turba,"{'gerente': 'R. Díaz', 'tel': '55-1111-0002'}",40.98,2026-06
3,1004,2026-04-13,S01,P009,7.0,32.03,Azúcar 1 kg,Abarrotes,33,Tlahuac Centro,Tlahuac,"{'gerente': 'M. López', 'tel': '55-1111-0001'}",224.21,2026-04
4,1005,2026-02-16,S04,P011,1.0,17.24,Jabón barra,Limpieza,18,Lago Tlahuac,Lago,"{'gerente': 'L. Vega', 'tel': '55-1111-0004'}",17.24,2026-02


## 5. Tres tablas resumen y ticket promedio

**Nota:** Los importes se calculan únicamente cuando `cantidad` y `precio_unitario` están disponibles. Se muestran las ventas sin importe para no confundir un valor desconocido con cero.

In [5]:
# Tabla 1: resumen por sucursal.
resumen_sucursal = (ventas_integradas.groupby(["sucursal_id", "nombre_sucursal"], dropna=False)
    .agg(ventas=("venta_id", "count"), unidades=("cantidad", "sum"),
         ventas_con_importe=("importe", "count"), ingreso=("importe", "sum"),
         ticket_promedio=("importe", "mean"))
    .sort_values("ingreso", ascending=False).reset_index())
print("TABLA 1. Ingresos y ticket promedio por sucursal")
display(resumen_sucursal)


TABLA 1. Ingresos y ticket promedio por sucursal


,sucursal_id,nombre_sucursal,ventas,unidades,ventas_con_importe,ingreso,ticket_promedio
0,S01,Tlahuac Centro,211,2342.0,197,75776.5,384.652284
1,S02,La Turba,170,1434.0,161,67364.29,418.411739
2,S03,Bosque de Tlahuac,129,868.0,118,31135.65,263.861441
3,S04,Lago Tlahuac,90,837.0,81,26750.23,330.249753


In [6]:
# Tabla 2: resumen por categoría de producto.
resumen_categoria = (ventas_integradas.groupby("categoria", dropna=False)
    .agg(ventas=("venta_id", "count"), unidades=("cantidad", "sum"),
         ventas_con_importe=("importe", "count"), ingreso=("importe", "sum"))
    .sort_values("ingreso", ascending=False).reset_index())
print("TABLA 2. Ingresos por categoría")
display(resumen_categoria)


TABLA 2. Ingresos por categoría


,categoria,ventas,unidades,ventas_con_importe,ingreso
0,Básicos,198,2557.0,184,88010.74
1,Abarrotes,143,990.0,134,56321.85
2,Bebidas,103,768.0,97,22909.84
3,Lácteos,56,694.0,52,19633.01
4,Limpieza,100,472.0,90,14151.23


In [7]:
# Tabla 3: tabla dinámica de ingresos por mes y sucursal.
tabla_mes_sucursal = ventas_integradas.pivot_table(
    index="mes", columns="nombre_sucursal", values="importe", aggfunc="sum", fill_value=0
).sort_index()
print("TABLA 3. Ingresos por mes × sucursal")
display(tabla_mes_sucursal)


TABLA 3. Ingresos por mes × sucursal


nombre_sucursal,Bosque de Tlahuac,La Turba,Lago Tlahuac,Tlahuac Centro
mes,,,,
2026-01,5396.45,4099.34,3755.03,4773.07
2026-02,2168.74,6066.4,2119.11,19817.3
2026-03,11040.19,5819.21,2287.57,6871.51
2026-04,5545.92,7412.49,3581.95,5209.65
2026-05,3666.79,38272.46,1298.88,31380.0
2026-06,3317.56,5694.39,13707.69,7724.97


## 6. Reporte de resultados y tres conclusiones

In [8]:
# Construir conclusiones con cifras obtenidas de los archivos reales.
mejor_sucursal = resumen_sucursal.iloc[0]
mejor_categoria = resumen_categoria.iloc[0]
serie_mensual = ventas_integradas.groupby("mes")["importe"].sum().sort_index()
mes_maximo = serie_mensual.idxmax()
mes_minimo = serie_mensual.idxmin()

conclusiones = [
    f"1. La sucursal con mayor ingreso registrado fue {mejor_sucursal['nombre_sucursal']}, con ${mejor_sucursal['ingreso']:,.2f} y un ticket promedio calculable de ${mejor_sucursal['ticket_promedio']:,.2f}.",
    f"2. La categoría con mayor ingreso registrado fue {mejor_categoria['categoria']}, con ${mejor_categoria['ingreso']:,.2f} en {int(mejor_categoria['ventas'])} ventas.",
    f"3. El mes con mayor ingreso registrado fue {mes_maximo} (${serie_mensual.max():,.2f}) y el menor fue {mes_minimo} (${serie_mensual.min():,.2f})."
]
for conclusion in conclusiones:
    print(conclusion)


1. La sucursal con mayor ingreso registrado fue Tlahuac Centro, con $75,776.50 y un ticket promedio calculable de $384.65.
2. La categoría con mayor ingreso registrado fue Básicos, con $88,010.74 en 198 ventas.
3. El mes con mayor ingreso registrado fue 2026-05 ($74,618.13) y el menor fue 2026-01 ($18,023.89).


### Tipo de fusión utilizado

Se utilizó **`left merge`** desde la tabla de ventas hacia los catálogos, con `validate="many_to_one"`: cada venta conserva su registro y se añade información de un único producto y una única sucursal. La comprobación con `indicator=True` permite detectar claves sin correspondencia. Un **`inner merge`** habría descartado ventas cuyos identificadores no aparecieran en los catálogos, mientras que un **`outer merge`** podría añadir filas de productos o sucursales sin ventas, cambiando la unidad de análisis. Por eso el `left merge` es el adecuado para responder preguntas sobre las ventas registradas.

### Limitaciones del análisis

- El importe no puede calcularse en ventas sin cantidad o precio; las sumas y promedios excluyen esos importes faltantes.
- Los valores extremos de cantidad se mantienen por falta de evidencia que permita corregirlos, y podrían influir considerablemente en los ingresos.
- Se eliminan únicamente duplicados exactos, no registros que solo compartan una clave.
- Se corrigió previamente la sintaxis del JSON proporcionado, sin alterar la identidad de las sucursales.

## 7. Verificación final y bitácora

In [9]:
# Aserciones de integridad y consistencia.
assert len(ventas_integradas) == len(ventas.drop_duplicates())
assert ventas_integradas["producto_id"].notna().all()
assert ventas_integradas["sucursal_id"].notna().all()
assert ventas_integradas["fecha"].notna().all()
assert int(resumen_sucursal["ventas"].sum()) == len(ventas_integradas)
assert int(resumen_categoria["ventas"].sum()) == len(ventas_integradas)

print("Validaciones superadas.")
print("Filas originales:", len(ventas))
print("Duplicados exactos eliminados:", int(ventas.duplicated().sum()))
print("Filas finales:", len(ventas_integradas))
print("Ventas con importe no calculable:", int(ventas_integradas["importe"].isna().sum()))


Validaciones superadas.
Filas originales: 612
Duplicados exactos eliminados: 12
Filas finales: 600
Ventas con importe no calculable: 43


| Etapa | Acción | Justificación |
|---|---|---|
| Lectura | CSV de ventas/productos y JSON de sucursales | Integrar fuentes heterogéneas |
| Diagnóstico | Nulos, duplicados y claves | Detectar problemas antes de fusionar |
| Limpieza | Eliminar duplicados exactos | Evitar doble conteo |
| Normalización | Quitar espacios y unificar mayúsculas en IDs | Hacer coincidir claves equivalentes |
| Conversión | Convertir fechas, cantidad y precio | Permitir análisis por mes e importes |
| Faltantes | Conservar importes desconocidos como NA | Evitar inventar ventas |
| Integración | Dos `left merge` con `many_to_one` | Mantener el número de ventas |
| Agrupación | Resúmenes por sucursal, categoría y mes | Responder el estudio de caso |
| Validación | `assert` y comparación de conteos | Confirmar integridad del resultado |

**Entrega:** este notebook ejecutado contiene el diagrama, código, tres tablas resumen, conclusiones y explicación de las fusiones. Para el reporte de máximo dos páginas, se pueden trasladar las secciones 1, 5 y 6 a un documento.